# Ada in the loop — Qwen3 frontier test (self-contained)

No repo clone needed. These cells write the code into the runtime and clone only the **public** adaL for data.

**Runtime → Change runtime type → GPU** (A100→Qwen3-8B, L4→4B, T4→1.7B). Optional Colab secret `HF_TOKEN`.

In [ ]:
%%writefile prepare_data.py
"""
prepare_data.py — turn adaL's real enriched facts into training + benchmark data
for the Qwen3 frontier test. Validatable on CPU; no GPU, no API key.

Outputs (under ./data/):
  facts.jsonl : {text, word_roles}  -- NL fact + per-word Ada role (Arm B labels)
  qa.jsonl    : {question, answer, entity, slot}  -- QA over the facts (both arms)

Run:  python prepare_data.py --n 50000
"""

import os, json, argparse, subprocess, random

ADAL = "/tmp/adaL"
SRC = f"{ADAL}/benchmark/curriculum/agi/ada.jsonl"

# Ada universal schema (vendored from glyphh-ai/adaL, Apache-2.0)
UNIVERSAL_SCHEMA = {
    "entity": ["name", "kind", "subkind"],
    "perceptual": ["color", "size", "shape", "texture", "sound", "smell", "taste", "temperature"],
    "spatial": ["location", "origin", "direction"],
    "temporal": ["time", "duration", "age", "era", "frequency"],
    "relational": ["subject", "predicate", "object", "possessor", "agent", "patient", "instrument"],
    "quantitative": ["count", "magnitude", "unit", "ratio"],
    "epistemic": ["source", "certainty", "modality"],
}
ALL_ROLES = [f"{l}.{r}" for l, rs in UNIVERSAL_SCHEMA.items() for r in rs]

# slot -> natural-language question template
QUESTION = {
    "relational.object":  "What does {e} relate to?",
    "relational.predicate": "What does {e} do?",
    "entity.kind":        "What kind of thing is {e}?",
    "entity.subkind":     "What subtype is {e}?",
    "spatial.location":   "Where is {e}?",
}


def ensure():
    if not os.path.exists(SRC):
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/glyphh-ai/adaL", ADAL],
                       check=True, env={**os.environ, "GIT_TERMINAL_PROMPT": "0"})


def words(v, cap=4):
    return [t for t in str(v).lower().replace("_", "-").replace("-", " ").split() if t][:cap]


def render(rec):
    """Fact -> (text, [(word, role)], (entity, slot, value)) or None."""
    e = rec.get("entity") or {}; r = rec.get("relational") or {}
    name = e.get("name")
    if not name:
        return None
    nw = words(name)
    pairs = [(w, "entity.name") for w in nw]
    if r.get("predicate") and r.get("object"):
        pw, ow = words(r["predicate"]), words(r["object"])
        pairs += [(w, "relational.predicate") for w in pw] + [(w, "relational.object") for w in ow]
        slot, val = "relational.object", " ".join(ow)
    elif e.get("kind"):
        kw = words(e["kind"])
        pairs += [("is", "O"), ("a", "O")] + [(w, "entity.kind") for w in kw]
        slot, val = "entity.kind", " ".join(kw)
    else:
        return None
    text = " ".join(w for w, _ in pairs)
    return text, pairs, (" ".join(nw), slot, val)


def main(n, seed=0):
    ensure()
    os.makedirs("data", exist_ok=True)
    rng = random.Random(seed)
    facts, qa = [], []
    with open(SRC) as f:
        for line in f:
            if len(facts) >= n:
                break
            try:
                out = render(json.loads(line))
            except Exception:
                continue
            if not out:
                continue
            text, pairs, (ent, slot, val) = out
            facts.append({"text": text, "word_roles": pairs})
            if slot in QUESTION and val:
                qa.append({"question": QUESTION[slot].format(e=ent),
                           "answer": val, "entity": ent, "slot": slot})
    rng.shuffle(facts); rng.shuffle(qa)
    with open("data/facts.jsonl", "w") as fo:
        for r in facts: fo.write(json.dumps(r) + "\n")
    with open("data/qa.jsonl", "w") as fo:
        for r in qa: fo.write(json.dumps(r) + "\n")

    # report
    from collections import Counter
    role_counts = Counter(role for r in facts for _, role in r["word_roles"])
    print(f"wrote data/facts.jsonl ({len(facts)} facts), data/qa.jsonl ({len(qa)} questions)")
    print("role distribution:", dict(role_counts))
    print("sample fact:", facts[0])
    print("sample qa:", qa[0] if qa else "—")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--n", type=int, default=50000)
    main(ap.parse_args().n)


In [ ]:
%%writefile smoke_test.py
"""
smoke_test.py — validate the dual-objective pipeline on CPU with a tiny model.

Same logic the Qwen3 fine-tune uses (LM next-token loss + Ada-role token loss,
masked, on the prepared data), but a ~tiny GRU so it runs in seconds on CPU.
This proves the data->batches->dual-loss->step->eval path is correct before any
GPU spend. Run as CI smoke too.

Run:  python smoke_test.py
"""

import json, torch, torch.nn as nn
from prepare_data import ALL_ROLES

ROLES = ["O"] + ALL_ROLES
RID = {r: i for i, r in enumerate(ROLES)}
PAD = 0


def load(path="data/facts.jsonl", limit=2000):
    facts = [json.loads(l) for i, l in enumerate(open(path)) if i < limit]
    vocab = {"<pad>": 0, "<unk>": 1}
    for f in facts:
        for w, _ in f["word_roles"]:
            vocab.setdefault(w, len(vocab))
    return facts, vocab


def batchify(facts, vocab, T=12):
    X, R = [], []
    for f in facts:
        ids = [vocab.get(w, 1) for w, _ in f["word_roles"]][:T]
        rs = [RID[r] for _, r in f["word_roles"]][:T]
        ids += [PAD] * (T - len(ids)); rs += [-100] * (T - len(rs))  # -100 = ignore
        X.append(ids); R.append(rs)
    return torch.tensor(X), torch.tensor(R)


class Tiny(nn.Module):
    def __init__(self, V, d=64, n_roles=len(ROLES)):
        super().__init__()
        self.emb = nn.Embedding(V, d, padding_idx=PAD)
        self.gru = nn.GRU(d, d, batch_first=True)
        self.lm = nn.Linear(d, V)
        self.role = nn.Linear(d, n_roles)

    def forward(self, x):
        h, _ = self.gru(self.emb(x))
        return self.lm(h), self.role(h)


def main():
    facts, vocab = load()
    X, R = batchify(facts, vocab)
    n = int(0.9 * len(X)); Xtr, Rtr, Xva, Rva = X[:n], R[:n], X[n:], R[n:]
    model = Tiny(len(vocab))
    opt = torch.optim.Adam(model.parameters(), lr=3e-3)
    lossf = nn.CrossEntropyLoss(ignore_index=-100)
    lossf_pad = nn.CrossEntropyLoss(ignore_index=PAD)

    def step(Xb, Rb, lam=1.0, train=True):
        lm, role = model(Xb)
        # LM: predict next token
        lm_loss = lossf_pad(lm[:, :-1].reshape(-1, lm.size(-1)), Xb[:, 1:].reshape(-1))
        # ROLE: tag current token (the Ada-in-the-loop objective)
        role_loss = lossf(role.reshape(-1, role.size(-1)), Rb.reshape(-1))
        loss = lm_loss + lam * role_loss
        if train:
            opt.zero_grad(); loss.backward(); opt.step()
        return lm_loss.item(), role_loss.item()

    print("step  lm_loss  role_loss")
    for s in range(200):
        bi = torch.randint(0, len(Xtr), (64,))
        l, r = step(Xtr[bi], Rtr[bi])
        if s % 50 == 0 or s == 199:
            print(f"{s:4d}  {l:7.3f}  {r:7.3f}")

    # eval role tagging accuracy on val
    with torch.no_grad():
        _, role = model(Xva)
        pred = role.argmax(-1)
        mask = Rva != -100
        acc = (pred[mask] == Rva[mask]).float().mean().item()
    print(f"\nval role-tag accuracy: {acc:.1%}")
    assert acc > 0.5, "pipeline produced no learning — smoke test FAILED"
    print("SMOKE TEST PASSED — dual-objective pipeline logic is correct.")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile finetune_qwen.py
"""
finetune_qwen.py — Arm B: Ada IN the loop on a real Qwen3 (LoRA + role head).

Fine-tunes Qwen3 with the SAME dual objective validated in smoke_test.py, but on
a real model and real BPE tokenization:
  loss = causal_LM_loss  +  lambda * Ada_role_token_loss
A small token-classification head over the last hidden states predicts each
token's Ada slot; word-level role labels are aligned to BPE subwords via offset
mapping. Only LoRA adapters + the role head train.

>>> NOT runnable in the CPU sandbox — validated only on a Colab GPU. The pipeline
    LOGIC is what smoke_test.py verifies; this is the same logic at scale. <<<

Run (on Colab GPU, after prepare_data.py):
    python finetune_qwen.py --model Qwen/Qwen3-8B --steps 1500
"""

import os, json, argparse, math
import torch, torch.nn as nn
from prepare_data import ALL_ROLES

ROLES = ["O"] + ALL_ROLES
RID = {r: i for i, r in enumerate(ROLES)}


def pick_model():
    """Biggest Qwen3 that fits the detected GPU (QLoRA 4-bit)."""
    if not torch.cuda.is_available():
        return "Qwen/Qwen3-0.6B"
    gb = torch.cuda.get_device_properties(0).total_memory / 1e9
    if gb >= 38:  return "Qwen/Qwen3-8B"     # A100 40GB
    if gb >= 22:  return "Qwen/Qwen3-4B"     # L4 24GB
    return "Qwen/Qwen3-1.7B"                  # T4 16GB


def build_examples(tokenizer, path="data/facts.jsonl", n=20000, T=64):
    """Tokenize each fact; align word roles -> subword role labels via offsets."""
    exs = []
    for i, line in enumerate(open(path)):
        if i >= n: break
        rec = json.loads(line)
        wr = rec["word_roles"]
        text = " ".join(w for w, _ in wr)
        enc = tokenizer(text, return_offsets_mapping=True, truncation=True, max_length=T)
        # map char span -> word index -> role
        spans, c = [], 0
        for w, r in wr:
            spans.append((c, c + len(w), RID[r])); c += len(w) + 1
        roles = []
        for (a, b) in enc["offset_mapping"]:
            if a == b:           # special token
                roles.append(-100); continue
            rid = -100
            for (sa, sb, rr) in spans:
                if a >= sa and a < sb:
                    rid = rr; break
            roles.append(rid)
        exs.append({"input_ids": enc["input_ids"],
                    "attention_mask": enc["attention_mask"], "roles": roles})
    return exs


class AdaQwen(nn.Module):
    """Qwen3 (LoRA) + a token-classification head for Ada roles."""
    def __init__(self, base, hidden, n_roles, lam=0.5):
        super().__init__()
        self.base = base
        self.role_head = nn.Linear(hidden, n_roles)
        self.lam = lam

    def forward(self, input_ids, attention_mask, labels=None, role_labels=None):
        out = self.base(input_ids=input_ids, attention_mask=attention_mask,
                        output_hidden_states=True)
        lm_logits = out.logits
        role_logits = self.role_head(out.hidden_states[-1].float())
        loss = lm_loss = role_loss = None
        if labels is not None:
            ce = nn.CrossEntropyLoss(ignore_index=-100)
            lm_loss = ce(lm_logits[:, :-1].reshape(-1, lm_logits.size(-1)),
                         labels[:, 1:].reshape(-1))
            role_loss = ce(role_logits.reshape(-1, role_logits.size(-1)),
                           role_labels.reshape(-1))
            loss = lm_loss + self.lam * role_loss
        return loss, lm_logits, role_logits, lm_loss, role_loss


def collate(batch, pad_id):
    T = max(len(b["input_ids"]) for b in batch)
    def pad(key, fill): return torch.tensor([b[key] + [fill]*(T-len(b[key])) for b in batch])
    ids = pad("input_ids", pad_id); am = pad("attention_mask", 0); roles = pad("roles", -100)
    labels = ids.clone(); labels[am == 0] = -100
    return ids, am, roles, labels


def main(model_name, steps, lam, n, lr):
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    model_name = model_name or pick_model()
    print(f"[device {dev}] model {model_name}  lambda {lam}")

    tok = AutoTokenizer.from_pretrained(model_name)
    if tok.pad_token is None: tok.pad_token = tok.eos_token
    qcfg = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                              bnb_4bit_compute_dtype=torch.bfloat16) if dev == "cuda" else None
    base = AutoModelForCausalLM.from_pretrained(
        model_name, quantization_config=qcfg, device_map="auto" if dev == "cuda" else None,
        torch_dtype=torch.bfloat16 if dev == "cuda" else torch.float32)
    if dev == "cuda": base = prepare_model_for_kbit_training(base)
    base = get_peft_model(base, LoraConfig(
        r=16, lora_alpha=32, lora_dropout=0.05, bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]))
    hidden = base.config.hidden_size
    model = AdaQwen(base, hidden, len(ROLES), lam=lam).to(dev)
    model.role_head.to(dev).to(torch.float32)

    exs = build_examples(tok, n=n)
    k = int(0.95*len(exs)); tr, va = exs[:k], exs[k:]
    opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=lr)

    import random
    model.train()
    for s in range(1, steps+1):
        batch = [tr[random.randrange(len(tr))] for _ in range(8)]
        ids, am, roles, labels = (t.to(dev) for t in collate(batch, tok.pad_token_id))
        loss, *_ , lml, rl = model(ids, am, labels, roles)
        opt.zero_grad(); loss.backward(); opt.step()
        if s % 100 == 0 or s == 1:
            print(f"step {s:4d}  loss {loss.item():.3f}  lm {lml.item():.3f}  role {rl.item():.3f}")

    # eval role-tag accuracy on val
    model.eval(); ok = tot = 0
    with torch.no_grad():
        for i in range(0, len(va), 8):
            b = va[i:i+8]
            ids, am, roles, labels = (t.to(dev) for t in collate(b, tok.pad_token_id))
            _, _, rlog, _, _ = model(ids, am, labels, roles)
            pred = rlog.argmax(-1); m = roles != -100
            ok += (pred[m] == roles[m]).sum().item(); tot += m.sum().item()
    print(f"\nval Ada role-tag accuracy: {ok/max(1,tot):.1%}  (lambda={lam})")
    print("Compare lambda>0 vs lambda=0: does the role objective hold at frontier scale,")
    print("and does it stay free on the LM loss?")


if __name__ == "__main__":
    ap = argparse.ArgumentParser()
    ap.add_argument("--model", default=None)
    ap.add_argument("--steps", type=int, default=1500)
    ap.add_argument("--lam", type=float, default=0.5)
    ap.add_argument("--n", type=int, default=20000)
    ap.add_argument("--lr", type=float, default=2e-4)
    a = ap.parse_args()
    main(a.model, a.steps, a.lam, a.n, a.lr)


In [ ]:
!pip install -q transformers peft accelerate bitsandbytes datasets
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - set Runtime->GPU')


In [ ]:
# data (clones PUBLIC glyphh-ai/adaL) + CPU smoke test
!python prepare_data.py --n 50000
!python smoke_test.py


In [ ]:
# Ada in the loop vs plain LM (same model/data/steps; only the role objective differs)
!python finetune_qwen.py --steps 1500 --lam 0.5
!python finetune_qwen.py --steps 1500 --lam 0.0


## Read it
- role-tag acc with lambda=0.5 >> chance(lambda=0) -> structure learned at frontier scale
- LM loss lambda=0.5 ≈ lambda=0 -> still free